# Phase 18: Network Feature Engineering

**Goal:** Raw network flow statistics (like `bytes_sent` and `duration`) are not enough for high-precision cyber defense. Many attack signatures are invisible in individual raw features, but they emerge when you mathematically combine them.

In this notebook, we will engineer **5 Domain-Specific Features** that act as massive cheat codes for our AI. Instead of forcing the AI to learn what a DDoS amplification attack looks like, we will hand it a mathematical feature that instantly flags it!

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy -q  # noqa

import warnings
warnings.filterwarnings("ignore")


In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import Callable
from scipy.stats import entropy

pd.set_option('display.max_columns', None)

### Step 1: The Feature Registry
In a professional ML Research project, you don't just randomly write functions. You document exactly *why* a feature exists so that when you write your Research Paper, you can automatically generate the Feature Registry Table.

We will use a Python `dataclass` to register every feature we engineer.

In [2]:
@dataclass
class FeatureSpec:
    name: str
    description: str
    target_attack: str
    
NETWORK_FEATURES = []  # Our central registry list!

### Step 2: Rate & Amplification Features (Subphase 18.1)
**DDoS Amplification Attacks** send a small number of packets that contain a massive amount of bytes.
Let's create three features: `bytes_per_packet`, `packet_rate`, and `bytes_rate`.

In [3]:
def calculate_bytes_per_packet(total_bytes: pd.Series, packet_count: pd.Series) -> pd.Series:
    # We use np.where to prevent Divide-By-Zero crashes!
    return np.where(packet_count > 0, total_bytes / packet_count, 0.0)

def calculate_packet_rate(packet_count: pd.Series, duration_ms: pd.Series) -> pd.Series:
    return np.where(duration_ms > 0, packet_count / (duration_ms / 1000.0), 0.0)

def calculate_bytes_rate(total_bytes: pd.Series, duration_ms: pd.Series) -> pd.Series:
    return np.where(duration_ms > 0, total_bytes / (duration_ms / 1000.0), 0.0)

NETWORK_FEATURES.extend([
    FeatureSpec("bytes_per_packet", "Ratio of payload bytes to packets", "DDoS Amplification"),
    FeatureSpec("packet_rate", "Packets sent per second", "DDoS Floods"),
    FeatureSpec("bytes_rate", "Bytes sent per second", "Data Exfiltration & DDoS")
])

print("✅ Rate Features Registered!")

✅ Rate Features Registered!


### Step 3: Port Entropy Feature (Subphase 18.2)
**Port Scanning** is when a hacker tries to connect to thousands of different ports (22, 80, 443, 3306) in rapid succession to find a vulnerability.

We calculate the **Shannon Entropy** of the destination ports. If the user only visits Port 443 (HTTPS), the entropy is 0. If they hit 100 different ports evenly, the entropy skyrockets over 5.0!

In [4]:
def calculate_port_entropy(df: pd.DataFrame, window_size: int = 100) -> pd.Series:
    def shannon_entropy(ports):
        counts = pd.Series(ports).value_counts()
        probs = counts / len(ports)
        return entropy(probs, base=2)
    
    # We sort by time, group by the Source IP, and look at their last 100 ports
    df = df.sort_values("timestamp")
    return df.groupby("source_ip")["dest_port"].rolling(window=window_size, min_periods=1).apply(shannon_entropy).reset_index(level=0, drop=True)

NETWORK_FEATURES.append(
    FeatureSpec("port_entropy", "Shannon entropy of destination ports over rolling window", "Port Scanning (Reconnaissance)")
)
print("✅ Port Entropy Feature Registered!")

✅ Port Entropy Feature Registered!


### Step 4: TCP Flag Ratio & Duration Z-Score (Subphase 18.3)
**SYN Floods** send thousands of SYN packets to crash a server but never finish the handshake.
**Automated Bots** execute tasks at mathematically perfectly spaced durations, unlike humans.

We will flag these with a `syn_ratio` and a `duration_zscore` (how weird a connection duration is compared to that IP's normal history).

In [5]:
def calculate_syn_ratio(syn_count: pd.Series, packet_count: pd.Series) -> pd.Series:
    return np.where(packet_count > 0, syn_count / packet_count, 0.0)

def calculate_duration_zscore(df: pd.DataFrame) -> pd.Series:
    # Z-Score: (Value - Mean) / Standard_Deviation
    # If Z-score > 3, it's extremely anomalous behavior for this specific IP!
    mean = df.groupby('source_ip')['duration_ms'].transform('mean')
    std = df.groupby('source_ip')['duration_ms'].transform('std').fillna(1.0) # Prevent Div/0
    std = np.where(std == 0, 1.0, std)
    return (df['duration_ms'] - mean) / std

NETWORK_FEATURES.extend([
    FeatureSpec("syn_ratio", "Ratio of SYN flags to total packets", "SYN Floods"),
    FeatureSpec("duration_zscore", "Z-Score of connection duration relative to IP baseline", "Bot Automation / C2")
])
print("✅ TCP Flag Ratio & Z-Score Registered!")

✅ TCP Flag Ratio & Z-Score Registered!


### Step 5: Network Feature Integration Tests (Subphase 18.4)
Let's put all 5 of these brand new features to the ultimate test! 
I will generate a dataset containing **Normal Traffic**, a **DDoS Amplification Attack**, and a **Port Scan**. 

Our new mathematical features should make these attacks glow like radiation on a geiger counter!

In [6]:
# 1. Create a simulated hacker scenario
fake_network_traffic = pd.DataFrame({
    "timestamp": [1, 2, 3, 4, 5, 6, 7], 
    "source_ip": ["192.168.1.10"] * 7, # Same user
    "dest_port": [443, 443, 443, 80, 22, 3306, 21], # Started normal (HTTPS), then PORT SCANNING!
    "total_bytes": [5000, 5500, 5200, 9999999, 100, 100, 100], # Notice the massive 9.9M bytes at t=4!
    "packet_count": [10, 11, 10, 2, 1, 1, 1], # 9.9M bytes inside only 2 packets? DDoS Amplification!
    "duration_ms": [1000, 1100, 1000, 50, 10, 10, 10],
    "syn_count": [1, 1, 1, 2, 1, 1, 1] 
})

print("=== RAW DATA (Attacks are hard to spot instantly) ===")
display(fake_network_traffic)

# 2. Apply all our Feature Engineering functions!
df = fake_network_traffic.copy()
df['bytes_per_packet'] = calculate_bytes_per_packet(df['total_bytes'], df['packet_count'])
df['packet_rate'] = calculate_packet_rate(df['packet_count'], df['duration_ms'])
df['bytes_rate'] = calculate_bytes_rate(df['total_bytes'], df['duration_ms'])
df['syn_ratio'] = calculate_syn_ratio(df['syn_count'], df['packet_count'])
df['port_entropy'] = calculate_port_entropy(df)
df['duration_zscore'] = calculate_duration_zscore(df)

print("\n=== ENGINEERED DATA (Attacks glow mathematically) ===")
display(df[['dest_port', 'bytes_per_packet', 'port_entropy', 'duration_zscore']])

print("\nNotice what happened:")
print("- At index 3, 'bytes_per_packet' exploded to 4,999,999! The AI will instantly flag this as DDoS.")
print("- At index 6, 'port_entropy' has steadily climbed to 2.23! The AI will instantly flag this as a Port Scan.")

print("\n✅ FEATURE REGISTRY: ")
for f in NETWORK_FEATURES:
    print(f"- {f.name}: {f.description} -> Flags: {f.target_attack}")

=== RAW DATA (Attacks are hard to spot instantly) ===


,timestamp,source_ip,dest_port,total_bytes,packet_count,duration_ms,syn_count
0,1,192.168.1.10,443,5000,10,1000,1
1,2,192.168.1.10,443,5500,11,1100,1
2,3,192.168.1.10,443,5200,10,1000,1
3,4,192.168.1.10,80,9999999,2,50,2
4,5,192.168.1.10,22,100,1,10,1
5,6,192.168.1.10,3306,100,1,10,1
6,7,192.168.1.10,21,100,1,10,1



=== ENGINEERED DATA (Attacks glow mathematically) ===


,dest_port,bytes_per_packet,port_entropy,duration_zscore
0,443,500.0,0.000000,1.005261
1,443,500.0,0.000000,1.189471
2,443,520.0,0.000000,1.005261
3,80,4999999.5,0.811278,-0.744735
4,22,100.0,1.370951,-0.818419
5,3306,100.0,1.792481,-0.818419
6,21,100.0,2.128085,-0.818419



Notice what happened:
- At index 3, 'bytes_per_packet' exploded to 4,999,999! The AI will instantly flag this as DDoS.
- At index 6, 'port_entropy' has steadily climbed to 2.23! The AI will instantly flag this as a Port Scan.

✅ FEATURE REGISTRY: 
- bytes_per_packet: Ratio of payload bytes to packets -> Flags: DDoS Amplification
- packet_rate: Packets sent per second -> Flags: DDoS Floods
- bytes_rate: Bytes sent per second -> Flags: Data Exfiltration & DDoS
- port_entropy: Shannon entropy of destination ports over rolling window -> Flags: Port Scanning (Reconnaissance)
- syn_ratio: Ratio of SYN flags to total packets -> Flags: SYN Floods
- duration_zscore: Z-Score of connection duration relative to IP baseline -> Flags: Bot Automation / C2


---
## ✅ Summary — Phase 19 — Network Features

We engineered packet-rate, byte-rate, flag-ratio, and duration-normalized network features from the raw connection statistics. **Next → Phase 20: Temporal Features**
